## Basic Implementation with Multiple Features

In [1]:
import math
import random

class MultipleLinearRegression:
    def __init__(self):
        self.weights = []      # Coefficients for each feature (w1, w2, ..., wn)
        self.intercept = 0.0   # Bias term (b)
    
    def fit_normal_equation(self, X, y):
        """
        Train using Normal Equation: w = (X^T X)^-1 X^T y
        X: List of lists [[x1, x2, ...], [x1, x2, ...], ...]
        y: List of target values
        """
        n_samples = len(X)
        n_features = len(X[0])
        
        # Add column of ones for intercept term
        X_design = [[1] + row for row in X]  # [[1, x1, x2, ...], ...]
        
        # Transpose
        XT = [[X_design[j][i] for j in range(n_samples)] for i in range(n_features + 1)]
        
        # X^T * X
        XTX = [[sum(XT[i][k] * X_design[k][j] for k in range(n_samples)) 
                for j in range(n_features + 1)] for i in range(n_features + 1)]
        
        # Inverse of XTX (using Gauss-Jordan elimination)
        XTX_inv = self._matrix_inverse(XTX)
        
        # X^T * y
        XTy = [sum(XT[i][j] * y[j] for j in range(n_samples)) for i in range(n_features + 1)]
        
        # w = (X^T X)^-1 * X^T y
        weights = [sum(XTX_inv[i][j] * XTy[j] for j in range(n_features + 1)) 
                   for i in range(n_features + 1)]
        
        self.intercept = weights[0]
        self.weights = weights[1:]
        
        return self
    
    def _matrix_inverse(self, matrix):
        """Gauss-Jordan elimination for matrix inversion"""
        n = len(matrix)
        # Augment with identity matrix
        aug = [row[:] + [1 if i == j else 0 for j in range(n)] for i, row in enumerate(matrix)]
        
        for i in range(n):
            # Find pivot
            pivot = aug[i][i]
            if pivot == 0:
                # Swap rows
                for k in range(i + 1, n):
                    if aug[k][i] != 0:
                        aug[i], aug[k] = aug[k], aug[i]
                        pivot = aug[i][i]
                        break
            
            # Normalize row
            for j in range(2 * n):
                aug[i][j] /= pivot
            
            # Eliminate other rows
            for k in range(n):
                if k != i:
                    factor = aug[k][i]
                    for j in range(2 * n):
                        aug[k][j] -= factor * aug[i][j]
        
        # Extract inverse
        return [row[n:] for row in aug]
    
    def fit_gradient_descent(self, X, y, learning_rate=0.01, epochs=1000):
        """Train using Gradient Descent"""
        n_samples = len(X)
        n_features = len(X[0])
        
        # Initialize weights randomly
        self.weights = [random.uniform(-0.1, 0.1) for _ in range(n_features)]
        self.intercept = 0.0
        self.cost_history = []
        
        for epoch in range(epochs):
            # Forward pass: make predictions
            predictions = self.predict(X)
            
            # Calculate gradients
            # dJ/dwj = (2/n) * sum((y_pred - y) * xj)
            gradient_weights = []
            for j in range(n_features):
                grad = (2 / n_samples) * sum((predictions[i] - y[i]) * X[i][j] 
                                              for i in range(n_samples))
                gradient_weights.append(grad)
            
            gradient_intercept = (2 / n_samples) * sum(predictions[i] - y[i] 
                                                        for i in range(n_samples))
            
            # Update parameters
            for j in range(n_features):
                self.weights[j] -= learning_rate * gradient_weights[j]
            self.intercept -= learning_rate * gradient_intercept
            
            # Track cost
            cost = self.compute_cost(X, y)
            self.cost_history.append(cost)
            
            if epoch % 200 == 0:
                print(f"Epoch {epoch:4d} | Cost: {cost:.6f}")
        
        return self
    
    def predict(self, X):
        """Make predictions for new data"""
        if isinstance(X[0], (int, float)):
            # Single sample
            return sum(w * x for w, x in zip(self.weights, X)) + self.intercept
        
        # Multiple samples
        return [sum(w * x for w, x in zip(self.weights, row)) + self.intercept 
                for row in X]
    
    def compute_cost(self, X, y):
        """Mean Squared Error"""
        predictions = self.predict(X)
        n = len(y)
        return sum((predictions[i] - y[i]) ** 2 for i in range(n)) / (2 * n)
    
    def r_squared(self, X, y):
        """R-squared score"""
        predictions = self.predict(X)
        y_mean = sum(y) / len(y)
        ss_res = sum((y[i] - predictions[i]) ** 2 for i in range(len(y)))
        ss_tot = sum((y[i] - y_mean) ** 2 for i in range(len(y)))
        return 1 - (ss_res / ss_tot)

# Example usage
if __name__ == "__main__":
    # Generate sample data with 3 features
    random.seed(42)
    n_samples = 100
    
    # True relationship: y = 2*x1 + 3*x2 - 1.5*x3 + 5 + noise
    X = [[random.uniform(0, 10) for _ in range(3)] for _ in range(n_samples)]
    y = [2*x[0] + 3*x[1] - 1.5*x[2] + 5 + random.gauss(0, 1.0) for x in X]
    
    print("=== Multiple Linear Regression from Scratch ===")
    print(f"Features: 3 (x1, x2, x3)")
    print(f"Samples: {n_samples}")
    
    # Train with Normal Equation
    model = MultipleLinearRegression()
    model.fit_normal_equation(X, y)
    print(f"\nNormal Equation Results:")
    print(f"  Intercept: {model.intercept:.4f}")
    print(f"  Weights: {[round(w, 4) for w in model.weights]}")
    print(f"  R² Score: {model.r_squared(X, y):.4f}")
    
    # Train with Gradient Descent
    model_gd = MultipleLinearRegression()
    model_gd.fit_gradient_descent(X, y, learning_rate=0.01, epochs=1000)
    print(f"\nGradient Descent Results:")
    print(f"  Intercept: {model_gd.intercept:.4f}")
    print(f"  Weights: {[round(w, 4) for w in model_gd.weights]}")
    print(f"  R² Score: {model_gd.r_squared(X, y):.4f}")

=== Multiple Linear Regression from Scratch ===
Features: 3 (x1, x2, x3)
Samples: 100

Normal Equation Results:
  Intercept: 4.5675
  Weights: [2.0746, 3.0043, -1.4843]
  R² Score: 0.9934
Epoch    0 | Cost: 137.704033
Epoch  200 | Cost: 0.887665
Epoch  400 | Cost: 0.691001
Epoch  600 | Cost: 0.590042
Epoch  800 | Cost: 0.538214

Gradient Descent Results:
  Intercept: 3.7499
  Weights: [2.1334, 3.0406, -1.4293]
  R² Score: 0.9930


## NumPy Implementation

In [2]:
import numpy as np

class MultipleLinearRegressionNumpy:
    def __init__(self):
        self.weights = None
        self.intercept = None
    
    def fit_normal_equation(self, X, y):
        """
        Normal Equation: w = (X^T X)^-1 X^T y
        """
        X = np.array(X)
        y = np.array(y)
        
        # Add column of ones for intercept
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        # w = (X^T X)^-1 X^T y
        weights = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        
        self.intercept = weights[0]
        self.weights = weights[1:]
        return self
    
    def fit_gradient_descent(self, X, y, learning_rate=0.01, epochs=1000):
        """
        Train using Gradient Descent (vectorized)
        """
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.random.randn(n_features) * 0.01
        self.intercept = 0.0
        self.cost_history = []
        
        for epoch in range(epochs):
            # Forward pass
            predictions = X @ self.weights + self.intercept
            
            # Gradients (vectorized!)
            dw = (2 / n_samples) * X.T @ (predictions - y)
            db = (2 / n_samples) * np.sum(predictions - y)
            
            # Update
            self.weights -= learning_rate * dw
            self.intercept -= learning_rate * db
            
            # Track cost
            cost = np.mean((predictions - y) ** 2) / 2
            self.cost_history.append(cost)
            
            if epoch % 200 == 0:
                print(f"Epoch {epoch:4d} | Cost: {cost:.6f}")
        
        return self
    
    def predict(self, X):
        X = np.array(X)
        return X @ self.weights + self.intercept
    
    def score(self, X, y):
        """R-squared score"""
        X = np.array(X)
        y = np.array(y)
        y_pred = self.predict(X)
        y_mean = np.mean(y)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y_mean) ** 2)
        return 1 - (ss_res / ss_tot)
    
    def get_params(self):
        """Return model parameters"""
        return {
            'weights': self.weights,
            'intercept': self.intercept
        }

# Example
if __name__ == "__main__":
    np.random.seed(42)
    
    # Generate data with 3 features
    n_samples = 200
    n_features = 3
    X = np.random.uniform(0, 10, (n_samples, n_features))
    true_weights = np.array([2.5, -1.8, 3.2])
    true_intercept = 7.0
    y = X @ true_weights + true_intercept + np.random.normal(0, 1.5, n_samples)
    
    print("=== NumPy Multiple Linear Regression ===")
    
    # Fit with Normal Equation
    model = MultipleLinearRegressionNumpy()
    model.fit_normal_equation(X, y)
    
    print(f"\nNormal Equation Results:")
    print(f"  True weights: {true_weights}")
    print(f"  Learned weights: {model.weights}")
    print(f"  True intercept: {true_intercept}")
    print(f"  Learned intercept: {model.intercept:.4f}")
    print(f"  R² Score: {model.score(X, y):.4f}")
    
    # Fit with Gradient Descent
    print("\nTraining with Gradient Descent:")
    model_gd = MultipleLinearRegressionNumpy()
    model_gd.fit_gradient_descent(X, y, learning_rate=0.001, epochs=1000)
    print(f"\nGradient Descent Results:")
    print(f"  Weights: {model_gd.weights}")
    print(f"  Intercept: {model_gd.intercept:.4f}")
    print(f"  R² Score: {model_gd.score(X, y):.4f}")

=== NumPy Multiple Linear Regression ===

Normal Equation Results:
  True weights: [ 2.5 -1.8  3.2]
  Learned weights: [ 2.50529852 -1.84470205  3.18821231]
  True intercept: 7.0
  Learned intercept: 7.1737
  R² Score: 0.9866

Training with Gradient Descent:
Epoch    0 | Cost: 410.163082
Epoch  200 | Cost: 3.713950
Epoch  400 | Cost: 3.418160
Epoch  600 | Cost: 3.221059
Epoch  800 | Cost: 3.041184

Gradient Descent Results:
  Weights: [ 2.84395443 -1.52072953  3.49083032]
  Intercept: 1.6905
  R² Score: 0.9666


## Scikit-Learn Implementation 

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd

# Create sample dataset
np.random.seed(42)
n_samples = 500

# Features: size, bedrooms, age, location_score, garden_size
X = np.random.uniform(0, 10, (n_samples, 5))
true_weights = np.array([50.0, 15.0, -2.5, 100.0, 5.0])
true_intercept = 100000

# Target: house price
y = X @ true_weights + true_intercept + np.random.normal(0, 5000, n_samples)

# Convert to DataFrame for better display
feature_names = ['size', 'bedrooms', 'age', 'location_score', 'garden_size']
df = pd.DataFrame(X, columns=feature_names)
df['price'] = y

print("=== Multiple Linear Regression with Scikit-Learn ===")
print(f"Dataset shape: {df.shape}")
print(f"Features: {feature_names}")
print("\nFirst 5 rows:")
print(df.head())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling (important for multiple features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate
print("\n=== Model Performance ===")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: ${mean_squared_error(y_test, y_pred, squared=False):,.2f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):,.2f}")

# Display coefficients
print("\n=== Model Coefficients ===")
for name, coef in zip(feature_names, model.coef_):
    print(f"  {name}: ${coef:,.2f} per unit")
print(f"  Intercept: ${model.intercept_:,.2f}")

# Feature importance (absolute coefficient values)
print("\n=== Feature Importance ===")
importance = np.abs(model.coef_)
for name, imp in sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True):
    print(f"  {name}: {imp:.2f}")

# Make predictions on new data
new_house = np.array([[1500, 3, 10, 7.5, 200]])  # size, bedrooms, age, location, garden
new_house_scaled = scaler.transform(new_house)
predicted_price = model.predict(new_house_scaled)
print(f"\nPredicted price for new house: ${predicted_price[0]:,.2f}")

=== Multiple Linear Regression with Scikit-Learn ===
Dataset shape: (500, 6)
Features: ['size', 'bedrooms', 'age', 'location_score', 'garden_size']

First 5 rows:
       size  bedrooms       age  location_score  garden_size          price
0  3.745401  9.507143  7.319939        5.986585     1.560186  103523.648915
1  1.559945  0.580836  8.661761        6.011150     7.080726  103927.651969
2  0.205845  9.699099  8.324426        2.123391     1.818250  103134.420348
3  1.834045  3.042422  5.247564        4.319450     2.912291  101018.629563
4  6.118529  1.394939  2.921446        3.663618     4.560700   99722.020151

=== Model Performance ===
R² Score: -0.0726
RMSE: $4,670.19
MAE: $3,674.70

=== Model Coefficients ===
  size: $-209.99 per unit
  bedrooms: $-293.86 per unit
  age: $101.17 per unit
  location_score: $310.36 per unit
  garden_size: $-215.55 per unit
  Intercept: $100,985.92

=== Feature Importance ===
  location_score: 310.36
  bedrooms: 293.86
  garden_size: 215.55
  size: 20

## With Feature Engineering and Interaction Terms

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Generate data with interactions
np.random.seed(42)
n_samples = 300

# Features
X = np.random.uniform(0, 10, (n_samples, 3))
# True relationship includes interaction term: y = 2*x1 + 3*x2 - x3 + 0.5*x1*x2 + noise
y = 2*X[:,0] + 3*X[:,1] - X[:,2] + 0.5*X[:,0]*X[:,1] + np.random.normal(0, 1.0, n_samples)

print("=== Multiple Linear Regression with Interaction Terms ===")

# Create pipeline with polynomial features (degree 2 includes interactions)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)),
    ('model', LinearRegression())
])

# Train
pipeline.fit(X, y)
y_pred = pipeline.predict(X)

print(f"R² Score: {r2_score(y, y_pred):.4f}")
print(f"Number of features (with interactions): {pipeline.named_steps['poly'].n_output_features_}")

# Show feature names
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
poly.fit(X)
feature_names = poly.get_feature_names_out(['x1', 'x2', 'x3'])
print("\nFeature names:")
for name, coef in zip(feature_names, pipeline.named_steps['model'].coef_):
    if abs(coef) > 0.01:  # Only show important ones
        print(f"  {name}: {coef:.4f}")

=== Multiple Linear Regression with Interaction Terms ===
R² Score: 0.9982
Number of features (with interactions): 9

Feature names:
  x1: 12.9485
  x2: 16.5581
  x3: -2.9166
  x1^2: 0.0136
  x1 x2: 4.3649
  x1 x3: 0.1576
  x2^2: -0.0531
  x2 x3: 0.0116
  x3^2: -0.0490


## Regularized Multiple Regression (Ridge & Lasso)

In [5]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.preprocessing import StandardScaler

# Generate high-dimensional data (many features)
np.random.seed(42)
n_samples = 200
n_features = 50

X = np.random.normal(0, 1, (n_samples, n_features))
# Only first 10 features are actually important
true_weights = np.zeros(n_features)
true_weights[:10] = np.random.uniform(1, 5, 10)
y = X @ true_weights + np.random.normal(0, 0.5, n_samples)

# Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== Regularized Multiple Linear Regression ===")
print(f"Features: {n_features} (only 10 are relevant)")

# 1. Ridge Regression (L2 regularization)
print("\n--- Ridge (L2) ---")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
print(f"R² (Train): {ridge.score(X_train_scaled, y_train):.4f}")
print(f"R² (Test): {ridge.score(X_test_scaled, y_test):.4f}")
print(f"Number of non-zero weights: {np.sum(np.abs(ridge.coef_) > 0.01)}")

# 2. Lasso Regression (L1 regularization - feature selection)
print("\n--- Lasso (L1) ---")
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
print(f"R² (Train): {lasso.score(X_train_scaled, y_train):.4f}")
print(f"R² (Test): {lasso.score(X_test_scaled, y_test):.4f}")
print(f"Number of non-zero weights: {np.sum(np.abs(lasso.coef_) > 0.01)}")

# 3. Find best alpha with cross-validation
print("\n--- Ridge with Cross-Validation ---")
param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_cv = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)
print(f"Best alpha: {ridge_cv.best_params_['alpha']}")
print(f"Best R² (Test): {ridge_cv.score(X_test_scaled, y_test):.4f}")

# 4. ElasticNet (combines L1 and L2)
print("\n--- ElasticNet (L1 + L2) ---")
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
print(f"R² (Test): {elastic.score(X_test_scaled, y_test):.4f}")
print(f"Number of non-zero weights: {np.sum(np.abs(elastic.coef_) > 0.01)}")

=== Regularized Multiple Linear Regression ===
Features: 50 (only 10 are relevant)

--- Ridge (L2) ---
R² (Train): 0.9973
R² (Test): 0.9933
Number of non-zero weights: 39

--- Lasso (L1) ---
R² (Train): 0.9945
R² (Test): 0.9929
Number of non-zero weights: 10

--- Ridge with Cross-Validation ---
Best alpha: 0.1
Best R² (Test): 0.9937

--- ElasticNet (L1 + L2) ---
R² (Test): 0.9862
Number of non-zero weights: 21
